In [3]:
# ============================================================
# DATASET - CONTACT LENSES
# ============================================================

import pandas as pd

dados = [
    ['young', 'myope', 'no',  'reduced', 'none'],
    ['young', 'myope', 'no',  'normal',  'soft'],
    ['young', 'myope', 'yes', 'reduced', 'none'],
    ['young', 'myope', 'yes', 'normal',  'hard'],
    ['young', 'hypermetrope', 'no',  'reduced', 'none'],
    ['young', 'hypermetrope', 'no',  'normal',  'soft'],
    ['young', 'hypermetrope', 'yes', 'reduced', 'none'],
    ['young', 'hypermetrope', 'yes', 'normal',  'hard'],

    ['pre-presbyopic', 'myope', 'no',  'reduced', 'none'],
    ['pre-presbyopic', 'myope', 'no',  'normal',  'soft'],
    ['pre-presbyopic', 'myope', 'yes', 'reduced', 'none'],
    ['pre-presbyopic', 'myope', 'yes', 'normal',  'hard'],
    ['pre-presbyopic', 'hypermetrope', 'no',  'reduced', 'none'],
    ['pre-presbyopic', 'hypermetrope', 'no',  'normal',  'soft'],
    ['pre-presbyopic', 'hypermetrope', 'yes', 'reduced', 'none'],
    ['pre-presbyopic', 'hypermetrope', 'yes', 'normal',  'none'],

    ['presbyopic', 'myope', 'no',  'reduced', 'none'],
    ['presbyopic', 'myope', 'no',  'normal',  'none'],
    ['presbyopic', 'myope', 'yes', 'reduced', 'none'],
    ['presbyopic', 'myope', 'yes', 'normal',  'hard'],
    ['presbyopic', 'hypermetrope', 'no',  'reduced', 'none'],
    ['presbyopic', 'hypermetrope', 'no',  'normal',  'soft'],
    ['presbyopic', 'hypermetrope', 'yes', 'reduced', 'none'],
    ['presbyopic', 'hypermetrope', 'yes', 'normal',  'none'],
]

colunas = [
    'Age',
    'Spectacle prescription',
    'Astigmatism',
    'Tear production rate',
    'Recommended lenses'
]

df = pd.DataFrame(dados, columns=colunas)

print("Dataset:")
display(df)


# ============================================================
# PARTE 1 - ALGORITMO 1R
# ============================================================

def one_rule(df, alvo):
    atributos = [col for col in df.columns if col != alvo]
    resultados = {}

    for atributo in atributos:
        print("\n" + "="*60)
        print(f"ATRIBUTO: {atributo}")
        print("="*60)

        freq = pd.crosstab(df[atributo], df[alvo])
        display(freq)

        regras = {}
        erro_total = 0

        for valor in freq.index:
            classe = freq.loc[valor].idxmax()
            acertos = freq.loc[valor].max()
            total = freq.loc[valor].sum()
            erros = total - acertos

            regras[valor] = classe
            erro_total += erros

            print(f"Se {atributo} = {valor} → {classe} | erros = {erros}")

        resultados[atributo] = {
            'regras': regras,
            'erro': erro_total
        }

        print(f"Erro total: {erro_total}")

    melhor = min(resultados, key=lambda x: resultados[x]['erro'])

    print("\n######## MELHOR REGRA (1R) ########")
    print(f"Atributo escolhido: {melhor}")
    for v, c in resultados[melhor]['regras'].items():
        print(f"Se {melhor} = {v} → {c}")

    return melhor, resultados[melhor], resultados


print("\n######## EXECUTANDO 1R ########")
melhor_1r, regra_1r, resultados_1r = one_rule(df, 'Recommended lenses')


# ============================================================
# PARTE 2 - ALGORITMO PRISM
# ============================================================

def prism_para_classe(df, classe_alvo, alvo='Recommended lenses'):
    regras = []
    dados_restantes = df.copy()
    atributos = [col for col in df.columns if col != alvo]

    while not dados_restantes[dados_restantes[alvo] == classe_alvo].empty:
        print("\n" + "="*60)
        print(f"Construindo regra para a classe: {classe_alvo}")
        print("="*60)

        subconjunto = dados_restantes.copy()
        condicoes = []
        atributos_disponiveis = atributos.copy()

        while True:
            melhor_precisao = -1
            melhor_condicao = None
            melhor_subconjunto = None

            total_positivos_antes = len(subconjunto[subconjunto[alvo] == classe_alvo])

            for atributo in atributos_disponiveis:
                for valor in subconjunto[atributo].unique():
                    filtrado = subconjunto[subconjunto[atributo] == valor]

                    if len(filtrado) == 0:
                        continue

                    positivos = len(filtrado[filtrado[alvo] == classe_alvo])
                    precisao = positivos / len(filtrado)

                    # desempate: se precisão igual, prefere maior número de positivos
                    if (
                        precisao > melhor_precisao or
                        (
                            precisao == melhor_precisao and
                            melhor_subconjunto is not None and
                            positivos > len(melhor_subconjunto[melhor_subconjunto[alvo] == classe_alvo])
                        )
                    ):
                        melhor_precisao = precisao
                        melhor_condicao = (atributo, valor)
                        melhor_subconjunto = filtrado

            # segurança: se não encontrou condição válida, interrompe
            if melhor_condicao is None:
                print("Nenhuma condição válida encontrada. Encerrando regra.")
                break

            atributo_escolhido, valor_escolhido = melhor_condicao
            print(f"Condição escolhida: {atributo_escolhido} = {valor_escolhido} | precisão = {melhor_precisao:.2f}")

            # segurança: se a condição não refinou nada, interrompe
            if len(melhor_subconjunto) == len(subconjunto):
                print("A condição não refinou o subconjunto. Encerrando para evitar loop.")
                break

            condicoes.append(melhor_condicao)
            subconjunto = melhor_subconjunto
            atributos_disponiveis.remove(atributo_escolhido)

            # se o subconjunto ficou puro para a classe alvo, finaliza a regra
            if all(subconjunto[alvo] == classe_alvo):
                print("Regra pura encontrada.")
                break

            # segurança extra: se não sobrar atributo para especializar
            if not atributos_disponiveis:
                print("Não há mais atributos disponíveis para refinar.")
                break

            total_positivos_depois = len(subconjunto[subconjunto[alvo] == classe_alvo])
            if total_positivos_depois == 0 or total_positivos_depois > total_positivos_antes:
                print("Refinamento inconsistente. Encerrando regra.")
                break

        # só salva a regra se ela cobrir pelo menos um exemplo da classe alvo
        exemplos_cobertos_alvo = subconjunto[subconjunto[alvo] == classe_alvo]

        if len(condicoes) > 0 and len(exemplos_cobertos_alvo) > 0:
            regras.append((condicoes, classe_alvo))

            # remove apenas os exemplos da classe alvo cobertos pela regra
            indices_remover = exemplos_cobertos_alvo.index
            dados_restantes = dados_restantes.drop(indices_remover)
        else:
            print("Não foi possível formar uma nova regra útil para essa classe.")
            break

    return regras


def prism(df, alvo='Recommended lenses'):
    todas_regras = []
    classes = df[alvo].unique()

    for classe in classes:
        regras_classe = prism_para_classe(df, classe, alvo)
        todas_regras.extend(regras_classe)

    return todas_regras


print("\n######## EXECUTANDO PRISM ########")
regras_prism = prism(df)


# ============================================================
# RESULTADO FINAL PRISM
# ============================================================

print("\n######## REGRAS FINAIS (PRISM) ########\n")

for i, (conds, classe) in enumerate(regras_prism, start=1):
    regra = " AND ".join([f"{a} = {v}" for a, v in conds])
    print(f"Regra {i}: SE {regra} → {classe}")

Dataset:


,Age,Spectacle prescription,Astigmatism,Tear production rate,Recommended lenses
0,young,myope,no,reduced,none
1,young,myope,no,normal,soft
2,young,myope,yes,reduced,none
3,young,myope,yes,normal,hard
4,young,hypermetrope,no,reduced,none
5,young,hypermetrope,no,normal,soft
6,young,hypermetrope,yes,reduced,none
7,young,hypermetrope,yes,normal,hard
8,pre-presbyopic,myope,no,reduced,none
9,pre-presbyopic,myope,no,normal,soft



######## EXECUTANDO 1R ########

ATRIBUTO: Age


Recommended lenses,hard,none,soft
Age,,,
pre-presbyopic,1,5,2
presbyopic,1,6,1
young,2,4,2


Se Age = pre-presbyopic → none | erros = 3
Se Age = presbyopic → none | erros = 2
Se Age = young → none | erros = 4
Erro total: 9

ATRIBUTO: Spectacle prescription


...

Se Spectacle prescription = hypermetrope → none | erros = 4
Se Spectacle prescription = myope → none | erros = 5
Erro total: 9

ATRIBUTO: Astigmatism


Recommended lenses,hard,none,soft
Astigmatism,,,
no,0,7,5
yes,4,8,0


Se Astigmatism = no → none | erros = 5
Se Astigmatism = yes → none | erros = 4
Erro total: 9

ATRIBUTO: Tear production rate


Recommended lenses,hard,none,soft
Tear production rate,,,
normal,4,3,5
reduced,0,12,0


Se Tear production rate = normal → soft | erros = 7
Se Tear production rate = reduced → none | erros = 0
Erro total: 7

######## MELHOR REGRA (1R) ########
Atributo escolhido: Tear production rate
Se Tear production rate = normal → soft
Se Tear production rate = reduced → none

######## EXECUTANDO PRISM ########

Construindo regra para a classe: none
Condição escolhida: Tear production rate = reduced | precisão = 1.00
Regra pura encontrada.

Construindo regra para a classe: none
Condição escolhida: Age = presbyopic | precisão = 0.50
Condição escolhida: Tear production rate = normal | precisão = 0.50
A condição não refinou o subconjunto. Encerrando para evitar loop.

Construindo regra para a classe: none
Condição escolhida: Age = pre-presbyopic | precisão = 0.25
Condição escolhida: Spectacle prescription = hypermetrope | precisão = 0.50
Condição escolhida: Astigmatism = yes | precisão = 1.00
Regra pura encontrada.

Construindo regra para a classe: soft
Condição escolhida: Astigmatism = 